# 04 · XAI — 무엇이 '좋은 질문'을 만드는가

MLOps 파이프라인 단계. **요구사항 #7(Explainable AI)** 담당.

LLM-as-a-Judge가 매기는 **Question Depth Score**는 블랙박스다. 우리는 코치 응답에서
해석 가능한 수치 피처(`src/data_prep.featurize_response`)를 뽑아, 그 피처로 Depth를 예측하는
**대리 모델(surrogate)** 을 학습하고 **SHAP·계수·순열 중요도**로 "어떤 특성이 점수를 끌어올리는가"를 설명한다.

- 피처: `ends_with_question, user_overlap, second_person_refs, n_question_marks, resp_word_len, has_quote_mark`
- 가설(피드백 연결): **사용자 발화에 그라운딩(user_overlap)** 되고 **열린 질문(ends_with_question)** 일수록 좋고,
  **사용자와 무관한 인용(has_quote_mark + 낮은 overlap = 환각)** 은 점수를 떨어뜨린다.

> 오프라인 모드는 피처→Depth의 **알려진 데이터 생성 과정**으로 라벨을 만들어 XAI 기법이
> 그 관계를 복원하는지 시연한다. 라이브 모드에선 Depth가 실제 Judge 점수로 대체된다.

In [ ]:
import sys, os, json, random
sys.path.append('..')
import numpy as np, pandas as pd
from src import data_prep as dp
random.seed(3); np.random.seed(3)
gs = json.load(open('../data/golden_set.json', encoding='utf-8'))
GOLDEN = gs['cases']
print('golden cases:', len(GOLDEN))

## 1. 응답 데이터셋 구성 + Feature Engineering

스타일별 특성을 가진 코치 응답을 만들고, `featurize_response()`로 수치 피처를 추출한다.
(전처리 노트북과 **동일한 FE 함수**를 재사용 — 일관성)

In [ ]:
def make_response(style, user_msg, fabricate=False):
    head = ' '.join(user_msg.split()[:3])  # 사용자 발화 앞부분 → 그라운딩에 사용
    if style == 'v2':
        txt = f'방금 "{head}"라고 하셨는데, 그 표현에서 당신이 전제한 것은 무엇인가요?'
    elif style == 'v3':
        txt = f'"{head}"라는 입장도 이해되지만, 반대로 이렇게 볼 수도 있지 않을까요?'
    else:  # v1
        txt = '좋은 감상이에요. 이 책에서 더 인상 깊었던 부분은 무엇인가요?'
        if fabricate:  # 사용자와 무관한 지어낸 인용(환각)
            txt += ' 특히 3장에서 주인공이 "운명은 없다"고 외치는 장면이 그렇죠.'
    return txt

rows = []
for case in GOLDEN:
    for style in ['v1','v2','v3']:
        fab = (style=='v1' and random.random() < 0.5)
        rows.append({'style':style,'user_msg':case['user_msg'],
                     'assistant_msg':make_response(style, case['user_msg'], fab),
                     'fabricated':int(fab)})
df = dp.featurize_frame(pd.DataFrame(rows))
FEATURES = ['ends_with_question','user_overlap','second_person_refs',
            'n_question_marks','resp_word_len','has_quote_mark']
print(df.groupby('style')[FEATURES].mean().round(2))
df.head(3)[['style']+FEATURES]

## 2. Depth Score 라벨

라이브: 실제 Judge 점수. 오프라인: 피처의 **알려진 함수 + 잡음**으로 라벨 생성
(그라운딩↑·열린질문↑ → 점수↑, 환각 인용 → 점수↓).

In [ ]:
USE_API = bool(os.getenv('OPENAI_API_KEY'))
if USE_API:
    # (간략) 라이브에서는 03 노트북의 judge를 재사용해 depth_score를 채우면 됨.
    # 데모에서는 오프라인 라벨 생성으로 통일.
    pass

def synth_depth(r):
    fab_penalty = 0.8 * r['has_quote_mark'] * (1.0 if r['user_overlap'] < 0.05 else 0.0)
    d = (1.6
         + 1.4 * r['ends_with_question']
         + 2.2 * r['user_overlap']
         + 0.12 * r['second_person_refs']
         + 0.10 * min(r['n_question_marks'], 3)
         - fab_penalty
         + np.random.normal(0, 0.25))
    return float(np.clip(d, 1, 5))

df['depth_score'] = df.apply(synth_depth, axis=1)
print('Depth by style:'); print(df.groupby('style')['depth_score'].mean().round(2))
print('환각 응답 평균 Depth:', round(df[df.fabricated==1]['depth_score'].mean(),2),
      '| 정상:', round(df[df.fabricated==0]['depth_score'].mean(),2))

## 3. 대리 모델 학습 (해석 가능한 surrogate)

선형회귀(계수 해석) + 그래디언트 부스팅(SHAP·비선형/상호작용)을 함께 사용한다.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.inspection import permutation_importance

X, y = df[FEATURES].astype(float), df['depth_score']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=3)

lin = LinearRegression().fit(Xtr, ytr)
gbr = GradientBoostingRegressor(random_state=3).fit(Xtr, ytr)
print('Linear R^2 :', round(r2_score(yte, lin.predict(Xte)),3))
print('GBR    R^2 :', round(r2_score(yte, gbr.predict(Xte)),3))

coef = pd.DataFrame({'feature':FEATURES,'linear_coef':lin.coef_.round(3)}) \
        .sort_values('linear_coef', key=abs, ascending=False)
print('\n선형 계수(부호=방향, 크기=영향):'); print(coef.to_string(index=False))

## 4. 순열 중요도 (Permutation Importance)

In [ ]:
pi = permutation_importance(gbr, Xte, yte, n_repeats=20, random_state=3)
imp = pd.DataFrame({'feature':FEATURES,'importance':pi.importances_mean.round(4)}) \
        .sort_values('importance', ascending=False)
print(imp.to_string(index=False))
import matplotlib.pyplot as plt
imp.set_index('feature')['importance'].sort_values().plot(kind='barh', color='#1F4E79', figsize=(7,3))
plt.title('Permutation importance (drives Question Depth)'); plt.tight_layout()
plt.savefig('xai_permutation_importance.png', dpi=120); plt.show()

## 5. SHAP — 개별 예측의 기여 분해

SHAP은 각 응답의 Depth 예측을 피처별 기여로 분해해, 전역(요약)·국소(단건) 설명을 모두 제공한다.

In [ ]:
import shap
explainer = shap.TreeExplainer(gbr)
sv = explainer.shap_values(Xte)
# 전역 중요도(평균 |SHAP|)
shap_imp = pd.DataFrame({'feature':FEATURES, 'mean_abs_shap':np.abs(sv).mean(0).round(4)}) \
            .sort_values('mean_abs_shap', ascending=False)
print('SHAP 전역 중요도:'); print(shap_imp.to_string(index=False))

plt.figure()
shap.summary_plot(sv, Xte, feature_names=FEATURES, show=False)
plt.tight_layout(); plt.savefig('xai_shap_summary.png', dpi=120, bbox_inches='tight'); plt.show()

In [ ]:
# 국소 설명 — 환각 응답 1건이 왜 낮은 점수를 받았는가
fab_idx = Xte.index[df.loc[Xte.index,'fabricated']==1]
if len(fab_idx):
    i = list(Xte.index).index(fab_idx[0])
    contrib = pd.DataFrame({'feature':FEATURES,'value':Xte.iloc[i].values,
                            'shap':sv[i].round(3)}).sort_values('shap')
    print('환각 응답 1건의 SHAP 기여(낮은 점수 원인 위→):')
    print(contrib.to_string(index=False))
    print('예측 Depth:', round(float(gbr.predict(Xte.iloc[[i]])[0]),2))

## 6. 결과 해석 + 산출물 저장

In [ ]:
top = shap_imp.iloc[0]['feature']
xai_summary = {
    'surrogate_r2': {'linear': round(float(r2_score(yte, lin.predict(Xte))),3),
                     'gbr': round(float(r2_score(yte, gbr.predict(Xte))),3)},
    'top_drivers': shap_imp.to_dict('records'),
    'linear_coef': coef.to_dict('records'),
    'interpretation': (
        '사용자 발화에 그라운딩(user_overlap)되고 열린 질문(ends_with_question)일수록 '
        'Depth가 오르며, 사용자와 무관한 인용(has_quote_mark+낮은 overlap=환각)은 점수를 떨어뜨린다. '
        '→ 프롬프트의 [그라운딩 규칙]과 정확히 일치하는 방향.'),
}
json.dump(xai_summary, open('xai_summary.json','w',encoding='utf-8'), ensure_ascii=False, indent=2)
print(json.dumps(xai_summary['top_drivers'], ensure_ascii=False, indent=2))
print('\n해석:', xai_summary['interpretation'])

## 7. 운영으로의 환류 (학습 없이)

XAI 결과는 **프롬프트 개선의 근거**가 된다(모델 학습 아님):
- `user_overlap`·`ends_with_question`이 핵심 동인 → 프롬프트의 '사용자 말에 닻을 내려라', 'Yes/No 금지' 규칙을 강화/유지.
- `has_quote_mark`가 환각과 함께 음(−)의 기여 → [그라운딩 규칙]의 '지어낸 인용 금지'가 정량적으로 정당화됨.
- 이 신호로 프롬프트 버전을 고치고 → 03 평가 노트북으로 회귀 검증 → 향상 시 버전 승격.